In [1]:
import numpy as np
import pandas as pd

from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier

from sklearn.ensemble import VotingClassifier

from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

In [2]:
data = load_breast_cancer()

X = data.data
y = data.target

# Convert to DataFrame (for better understanding)
df = pd.DataFrame(X, columns=data.feature_names)
df['target'] = y

print("Dataset Shape:", df.shape)
print("\nFirst 5 rows:\n", df.head())

Dataset Shape: (569, 31)

First 5 rows:
    mean radius  mean texture  mean perimeter  mean area  mean smoothness  \
0        17.99         10.38          122.80     1001.0          0.11840   
1        20.57         17.77          132.90     1326.0          0.08474   
2        19.69         21.25          130.00     1203.0          0.10960   
3        11.42         20.38           77.58      386.1          0.14250   
4        20.29         14.34          135.10     1297.0          0.10030   

   mean compactness  mean concavity  mean concave points  mean symmetry  \
0           0.27760          0.3001              0.14710         0.2419   
1           0.07864          0.0869              0.07017         0.1812   
2           0.15990          0.1974              0.12790         0.2069   
3           0.28390          0.2414              0.10520         0.2597   
4           0.13280          0.1980              0.10430         0.1809   

   mean fractal dimension  ...  worst texture  wors

In [3]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)


In [4]:
scaler = StandardScaler()

X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

In [5]:
log_model = LogisticRegression(max_iter=5000, random_state=42)
dt_model = DecisionTreeClassifier(random_state=42)
knn_model = KNeighborsClassifier(n_neighbors=5)

In [6]:
log_model.fit(X_train, y_train)
dt_model.fit(X_train, y_train)
knn_model.fit(X_train, y_train)

KNeighborsClassifier()

In [7]:
log_pred = log_model.predict(X_test)
dt_pred = dt_model.predict(X_test)
knn_pred = knn_model.predict(X_test)

In [8]:
print("\n===== Individual Model Performance =====")

print("\nLogistic Regression Accuracy:", accuracy_score(y_test, log_pred))
print("Confusion Matrix:\n", confusion_matrix(y_test, log_pred))
print("Classification Report:\n", classification_report(y_test, log_pred))

print("\nDecision Tree Accuracy:", accuracy_score(y_test, dt_pred))
print("Confusion Matrix:\n", confusion_matrix(y_test, dt_pred))
print("Classification Report:\n", classification_report(y_test, dt_pred))

print("\nKNN Accuracy:", accuracy_score(y_test, knn_pred))
print("Confusion Matrix:\n", confusion_matrix(y_test, knn_pred))
print("Classification Report:\n", classification_report(y_test, knn_pred))


===== Individual Model Performance =====

Logistic Regression Accuracy: 0.9824561403508771
Confusion Matrix:
 [[41  1]
 [ 1 71]]
Classification Report:
               precision    recall  f1-score   support

           0       0.98      0.98      0.98        42
           1       0.99      0.99      0.99        72

    accuracy                           0.98       114
   macro avg       0.98      0.98      0.98       114
weighted avg       0.98      0.98      0.98       114


Decision Tree Accuracy: 0.9122807017543859
Confusion Matrix:
 [[39  3]
 [ 7 65]]
Classification Report:
               precision    recall  f1-score   support

           0       0.85      0.93      0.89        42
           1       0.96      0.90      0.93        72

    accuracy                           0.91       114
   macro avg       0.90      0.92      0.91       114
weighted avg       0.92      0.91      0.91       114


KNN Accuracy: 0.956140350877193
Confusion Matrix:
 [[39  3]
 [ 2 70]]
Classification 

In [9]:
voting_clf_hard = VotingClassifier(
    estimators=[
        ('lr', log_model),
        ('dt', dt_model),
        ('knn', knn_model)
    ],
    voting='hard'
)

voting_clf_hard.fit(X_train, y_train)

hard_pred = voting_clf_hard.predict(X_test)

print("\n===== Ensemble Model (Hard Voting) =====")
print("Accuracy:", accuracy_score(y_test, hard_pred))
print("Confusion Matrix:\n", confusion_matrix(y_test, hard_pred))
print("Classification Report:\n", classification_report(y_test, hard_pred))


===== Ensemble Model (Hard Voting) =====
Accuracy: 0.9824561403508771
Confusion Matrix:
 [[40  2]
 [ 0 72]]
Classification Report:
               precision    recall  f1-score   support

           0       1.00      0.95      0.98        42
           1       0.97      1.00      0.99        72

    accuracy                           0.98       114
   macro avg       0.99      0.98      0.98       114
weighted avg       0.98      0.98      0.98       114



In [10]:
voting_clf_soft = VotingClassifier(
    estimators=[
        ('lr', log_model),
        ('dt', dt_model),
        ('knn', knn_model)
    ],
    voting='soft'
)

voting_clf_soft.fit(X_train, y_train)

soft_pred = voting_clf_soft.predict(X_test)

print("\n===== Ensemble Model (Soft Voting) =====")
print("Accuracy:", accuracy_score(y_test, soft_pred))
print("Confusion Matrix:\n", confusion_matrix(y_test, soft_pred))
print("Classification Report:\n", classification_report(y_test, soft_pred))


===== Ensemble Model (Soft Voting) =====
Accuracy: 0.9649122807017544
Confusion Matrix:
 [[39  3]
 [ 1 71]]
Classification Report:
               precision    recall  f1-score   support

           0       0.97      0.93      0.95        42
           1       0.96      0.99      0.97        72

    accuracy                           0.96       114
   macro avg       0.97      0.96      0.96       114
weighted avg       0.97      0.96      0.96       114



In [11]:
results = pd.DataFrame({
    'Model': ['Logistic Regression', 'Decision Tree', 'KNN', 'Voting (Hard)', 'Voting (Soft)'],
    'Accuracy': [
        accuracy_score(y_test, log_pred),
        accuracy_score(y_test, dt_pred),
        accuracy_score(y_test, knn_pred),
        accuracy_score(y_test, hard_pred),
        accuracy_score(y_test, soft_pred)
    ]
})

print("\n===== Model Comparison =====")
print(results)


===== Model Comparison =====
                 Model  Accuracy
0  Logistic Regression  0.982456
1        Decision Tree  0.912281
2                  KNN  0.956140
3        Voting (Hard)  0.982456
4        Voting (Soft)  0.964912


In [12]:
results.to_csv("ensemble_results.csv", index=False)